# Boundary Diagnostic — PPS Radius Calibration

Xác định chính xác các window có `boundary_flag = True` để lập kế hoạch rerun
riêng từng trường hợp với rho range rộng hơn. Notebook chỉ đọc các CSV đã có;
không chạy lại PPS optimization và không lưu thêm artifact.

In [1]:
# TASK 1 - Load and merge all calibration CSV files
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root():
    """Locate the repository root."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "phase1" / "results").is_dir():
            return candidate
    raise FileNotFoundError("Repository root was not found.")


PROJECT_ROOT = find_project_root()
CSV_DIR = (
    PROJECT_ROOT
    / "phase1"
    / "results"
    / "pps"
    / "pps_radius_calibration"
    / "csv"
)
CSV_PATTERN = "pps_radius_session_*.csv"
EXPECTED_SESSION_COUNT = 20
CALIBRATION_COLUMNS = [
    "session",
    "representation",
    "state",
    "window_size",
    "window_id",
    "rho_star",
    "rho_NN",
    "peak_C2",
    "peak_C2_sd",
    "boundary_flag",
]

csv_files = sorted(CSV_DIR.glob(CSV_PATTERN))
if len(csv_files) != EXPECTED_SESSION_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_SESSION_COUNT} CSV files, "
        f"found {len(csv_files)}."
    )

frames = []
for csv_path in csv_files:
    frame = pd.read_csv(csv_path)
    missing = set(CALIBRATION_COLUMNS).difference(frame.columns)
    if missing:
        raise ValueError(
            f"{csv_path.name} is missing {sorted(missing)}."
        )
    frames.append(frame[CALIBRATION_COLUMNS].copy())

calibration_data = pd.concat(frames, ignore_index=True)
if calibration_data[CALIBRATION_COLUMNS].isna().any().any():
    raise ValueError("Calibration data contain missing required values.")

total_windows = len(calibration_data)
total_boundary = int(calibration_data["boundary_flag"].sum())
boundary_fraction = total_boundary / total_windows
overview = pd.DataFrame(
    {
        "Metric": [
            "CSV files",
            "Unique sessions",
            "Total windows",
            "Boundary windows",
            "Boundary fraction",
        ],
        "Value": [
            len(csv_files),
            calibration_data["session"].nunique(),
            total_windows,
            total_boundary,
            boundary_fraction,
        ],
    }
)
overview

,Metric,Value
0,CSV files,20.000000
1,Unique sessions,20.000000
2,Total windows,3181.000000
3,Boundary windows,1.000000
4,Boundary fraction,0.000314


In [2]:
# TASK 2 - Extract and sort the boundary cases
BOUNDARY_SORT_COLUMNS = [
    "session",
    "representation",
    "state",
    "window_size",
    "window_id",
]
BOUNDARY_REPORT_COLUMNS = BOUNDARY_SORT_COLUMNS + [
    "rho_star",
    "rho_NN",
    "peak_C2",
    "peak_C2_sd",
]

boundary_cases = (
    calibration_data[calibration_data["boundary_flag"]]
    .sort_values(BOUNDARY_SORT_COLUMNS)
    .reset_index(drop=True)
)
if len(boundary_cases) != total_boundary:
    raise RuntimeError("Boundary extraction count is inconsistent.")

boundary_cases[BOUNDARY_REPORT_COLUMNS]

,session,representation,state,window_size,window_id,rho_star,rho_NN,peak_C2,peak_C2_sd
0,23,Processed,Awake,60,45,5.0,0.023801,401.0,183.926616


In [3]:
# TASK 3 - Summarize the boundary distribution
def summarize_factor(data, factor):
    """Count total and boundary windows for one factor."""
    total = data.groupby(factor, dropna=False).size().rename("Total_N")
    boundary = (
        data.groupby(factor, dropna=False)["boundary_flag"]
        .sum()
        .astype(int)
        .rename("Boundary_N")
    )
    summary = pd.concat([total, boundary], axis=1).reset_index()
    summary["Boundary_percent"] = (
        100.0 * summary["Boundary_N"] / summary["Total_N"]
    )
    summary.insert(0, "Factor", factor)
    summary = summary.rename(columns={factor: "Group"})
    return summary


boundary_distribution = pd.concat(
    [
        summarize_factor(calibration_data, factor)
        for factor in (
            "session",
            "representation",
            "state",
            "window_size",
        )
    ],
    ignore_index=True,
)

joint_boundary_counts = (
    boundary_cases.groupby(
        ["session", "representation", "state", "window_size"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Boundary_N"})
)

display(boundary_distribution)
joint_boundary_counts

,Factor,Group,Total_N,Boundary_N,Boundary_percent
0,session,1,105,0,0.000000
1,session,4,158,0,0.000000
2,session,5,176,0,0.000000
3,session,6,168,0,0.000000
4,session,7,166,0,0.000000
5,session,8,164,0,0.000000
6,session,9,233,0,0.000000
7,session,10,122,0,0.000000
8,session,11,132,0,0.000000
9,session,12,191,0,0.000000


,session,representation,state,window_size,Boundary_N
0,23,Processed,Awake,60,1


In [4]:
# TASK 4 - Identify lower and upper boundary hits
FROZEN_RANGES = {
    "Processed": (5.0, 200.0),
    "Raw": (5.0, 400.0),
}


def identify_boundary_side(row):
    """Classify a boundary optimum using its frozen rho grid."""
    rho_min, rho_max = FROZEN_RANGES[row["representation"]]
    if np.isclose(row["rho_star"], rho_min):
        return "lower_boundary"
    if np.isclose(row["rho_star"], rho_max):
        return "upper_boundary"
    return "unclassified"


grid_table = pd.DataFrame(
    [
        {
            "representation": representation,
            "rho_min": bounds[0],
            "rho_max": bounds[1],
        }
        for representation, bounds in FROZEN_RANGES.items()
    ]
)
boundary_cases["boundary_side"] = boundary_cases.apply(
    identify_boundary_side,
    axis=1,
)
unclassified = boundary_cases["boundary_side"].eq("unclassified")
if unclassified.any():
    raise ValueError(
        f"Found {int(unclassified.sum())} unclassified boundary cases."
    )

boundary_direction = boundary_cases[
    [
        "session",
        "representation",
        "state",
        "window_size",
        "window_id",
        "rho_star",
        "boundary_side",
    ]
]
display(grid_table)
boundary_direction

,representation,rho_min,rho_max
0,Processed,5.0,200.0
1,Raw,5.0,400.0


,session,representation,state,window_size,window_id,rho_star,boundary_side
0,23,Processed,Awake,60,45,5.0,lower_boundary


In [5]:
# TASK 5 - Create the targeted rerun plan
UPPER_EXPANSION_FACTOR = 2.0
LOWER_REDUCTION_FACTOR = 5.0


def recommend_new_range(row):
    """Expand only the grid side reached by the optimum."""
    rho_min, rho_max = FROZEN_RANGES[row["representation"]]
    if row["boundary_side"] == "upper_boundary":
        new_min = rho_min
        new_max = rho_max * UPPER_EXPANSION_FACTOR
    else:
        new_min = rho_min / LOWER_REDUCTION_FACTOR
        new_max = rho_max
    return pd.Series(
        {
            "recommended_rho_min": new_min,
            "recommended_rho_max": new_max,
            "recommended_new_range": f"{new_min:g}–{new_max:g}",
        }
    )


range_recommendations = boundary_cases.apply(
    recommend_new_range,
    axis=1,
)
rerun_plan = pd.concat(
    [boundary_cases.reset_index(drop=True), range_recommendations],
    axis=1,
).rename(columns={"rho_star": "old_rho_star"})
rerun_plan = rerun_plan[
    [
        "session",
        "representation",
        "state",
        "window_size",
        "window_id",
        "old_rho_star",
        "boundary_side",
        "recommended_new_range",
    ]
]
rerun_plan

,session,representation,state,window_size,window_id,old_rho_star,boundary_side,recommended_new_range
0,23,Processed,Awake,60,45,5.0,lower_boundary,1–200


In [6]:
# TASK 6 - Print the final diagnostic summary
affected_sessions = sorted(boundary_cases["session"].unique().tolist())
processed_count = int(
    boundary_cases["representation"].eq("Processed").sum()
)
raw_count = int(boundary_cases["representation"].eq("Raw").sum())
upper_count = int(
    boundary_cases["boundary_side"].eq("upper_boundary").sum()
)
lower_count = int(
    boundary_cases["boundary_side"].eq("lower_boundary").sum()
)

print(f"Total boundary cases: {total_boundary} / {total_windows}")
print(f"Boundary fraction: {boundary_fraction:.4%}")
print(f"Sessions affected: {affected_sessions}")
print(f"Processed: {processed_count} cases")
print(f"Raw: {raw_count} cases")
print(f"Upper-bound hits: {upper_count}")
print(f"Lower-bound hits: {lower_count}")
print(f"Windows requiring rerun: {len(rerun_plan)}")

diagnostic_summary = pd.DataFrame(
    {
        "Metric": [
            "Total boundary cases",
            "Affected sessions",
            "Processed cases",
            "Raw cases",
            "Upper-bound hits",
            "Lower-bound hits",
            "Windows requiring rerun",
        ],
        "Value": [
            f"{total_boundary} / {total_windows}",
            ", ".join(map(str, affected_sessions)),
            processed_count,
            raw_count,
            upper_count,
            lower_count,
            len(rerun_plan),
        ],
    }
)
diagnostic_summary

Total boundary cases: 1 / 3181
Boundary fraction: 0.0314%
Sessions affected: [23]
Processed: 1 cases
Raw: 0 cases
Upper-bound hits: 0
Lower-bound hits: 1
Windows requiring rerun: 1


,Metric,Value
0,Total boundary cases,1 / 3181
1,Affected sessions,23
2,Processed cases,1
3,Raw cases,0
4,Upper-bound hits,0
5,Lower-bound hits,1
6,Windows requiring rerun,1


## Decision

- `boundary_flag = False`: giữ nguyên `rho_star` hiện tại.
- `boundary_flag = True`: chỉ rerun window tương ứng với range được đề xuất.
- Giữ nguyên `m`, `tau`, `segment_length`, `count_mode`, trials và PPS core.
- Khi rerun đạt interior optimum: cập nhật calibration CSV rồi freeze lookup
  manifest cho surrogate testing.

Notebook diagnostic này không lưu danh sách hoặc bảng mới ra disk.